# vi-function-calling-slm · 01 sinh dữ liệu

Kaggle hoặc Colab, **CPU là đủ**, bật Internet. Kaggle: Settings → Accelerator → None (CPU). Colab: runtime CPU mặc định là được.
Input: Kaggle dùng dataset `vi-function-calling-slm-code` (code của repo); Colab tự clone repo từ GitHub,
dữ liệu sinh ra lưu vào Google Drive (`MyDrive/ai-portfolio/vi-function-calling-slm/data/gen`) cho notebook 02-04.
Secrets (Kaggle: Add-ons → Secrets; Colab: biểu tượng chìa khoá bên trái, bật Notebook access):
`GROQ_API_KEY` (bắt buộc), `HF_TOKEN` (tuỳ chọn, đẩy dữ liệu lên Hub).

Các bước:
1. Bốc seed (kịch bản, tool, giá trị slot, trạng thái xe, văn phong) bằng code
2. Groq `openai/gpt-oss-120b` viết câu người dùng cho từng seed (lô 8 seed / request)
3. Nhãn gold dựng từ seed, không do LLM gán
4. Lọc (schema, luật an toàn, câu có nhắc đủ giá trị), bỏ trùng, chia train/val/test theo group
5. Dựng cặp DPO từ nhiễu có chủ đích

Groq free tier giới hạn request/phút và token/ngày. Hết quota giữa chừng thì chạy lại cell sinh dữ liệu
(hoặc chạy lại notebook hôm sau): file raw ghi nối tiếp, seed đã có sẽ bỏ qua.

In [ ]:
import glob, json, os, sys

IN_KAGGLE = os.path.exists("/kaggle/input")
IN_COLAB = "google.colab" in sys.modules
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/vi_fc/__init__.py", recursive=True)[0].rsplit("/vi_fc", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vi-function-calling-slm" if IN_COLAB else os.path.abspath("vi-function-calling-slm")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vi-function-calling-slm {REPO}
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vi-function-calling-slm" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE
print(CODE)
!pip install -q python-dotenv


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

for k in ("GROQ_API_KEY", "HF_TOKEN"):
    v = secret(k)
    if v:
        os.environ[k] = v
    else:
        print("thiếu secret", k)

## Xem thử đề bài gửi LLM

In [ ]:
import random
from vi_fc.datagen.generate import SYSTEM, build_prompt
from vi_fc.datagen.seeds import sample_seed

rng = random.Random(0)
demo = [sample_seed(rng, i) for i in range(8)]
print(SYSTEM)
print(build_prompt(demo))

## Sinh 4000 seed
~500 request. Nếu log báo 429 liên tục / hết quota ngày thì dừng, mai chạy tiếp.

In [ ]:
!python -m vi_fc.datagen.generate --n 4000 --out data/gen/raw.jsonl --batch-size 8
!wc -l data/gen/raw.jsonl

## Lọc, bỏ trùng, chia split

In [ ]:
!python -m vi_fc.datagen.filters --raw data/gen/raw.jsonl --out data/gen

stats = json.load(open("data/gen/stats.json", encoding="utf-8"))
print("lý do bị loại:", json.dumps(stats["invalid_reasons"], ensure_ascii=False, indent=1))
for k, v in stats["splits"].items():
    print(k, v["n"], v["scenario"])
print(json.dumps(stats["splits"]["train"]["tools"], ensure_ascii=False))
print(json.dumps(stats["splits"]["train"]["style"], ensure_ascii=False))

## Đọc tay vài ví dụ mỗi kịch bản
Bước này quan trọng hơn mọi con số: đọc ~50 câu xem LLM có viết sai ý seed không.

In [ ]:
from collections import defaultdict

train = [json.loads(l) for l in open("data/gen/train.jsonl", encoding="utf-8")]
by_sc = defaultdict(list)
for r in train:
    by_sc[r["scenario"]].append(r)
for sc, rs in by_sc.items():
    print(f"=== {sc} ({len(rs)})")
    for r in random.Random(1).sample(rs, min(6, len(rs))):
        hist = " | ".join(m["content"] for m in r["history"])
        print(f"  [{r['style']}] {hist + ' || ' if hist else ''}{r['user']}")
        print("     ->", json.dumps(r["gold"], ensure_ascii=False)[:160])

## Câu bị loại vì "không nhắc giá trị"
Xem để biết bộ lọc có quá chặt không (LLM dùng từ đồng nghĩa mà bảng alias chưa có).

In [ ]:
from vi_fc.datagen.filters import check_record

raw = [json.loads(l) for l in open("data/gen/raw.jsonl", encoding="utf-8")]
miss = [(r, check_record(r)) for r in raw]
miss = [(r, p) for r, p in miss if p and "không nhắc" in p[0]]
print(len(miss), "câu bị loại vì không nhắc giá trị")
for r, p in miss[:25]:
    print(r["user_raw"], "  <-", p[0])

## Cặp DPO (nhiễu có chủ đích)

In [ ]:
!python -m vi_fc.dpo --data data/gen/train.jsonl --out data/gen/dpo_train.jsonl --per-record 1
p = json.loads(open("data/gen/dpo_train.jsonl", encoding="utf-8").readline())
print(p["type"]); print("chosen  :", p["chosen"]); print("rejected:", p["rejected"])

## (tuỳ chọn) đẩy dữ liệu lên HF Hub
Output của gpt-oss-120b (Apache-2.0) được phép dùng để train và chia sẻ.

In [ ]:
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi

    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo("DuongCodeAI/vi-fc-synth", repo_type="dataset", exist_ok=True)
    api.upload_folder(folder_path="data/gen", repo_id="DuongCodeAI/vi-fc-synth", repo_type="dataset",
                      allow_patterns=["train.jsonl", "val.jsonl", "test.jsonl", "dpo_train.jsonl", "stats.json"])